>NOTE: This is notebook is for exploration before creating the final ingestion module, which will be a Python script

## Ingestion Strategy

### 1. Loop over each wdl file in `data/wdl/`, storing information as a dictionary of dictionaries like so:

```{python}
wdl_library = {'tool1': 
                  {'task1': 'entire wdl task', 
                   'task2': 'entire task'},
               'tool2': {...}, 
               ...}
```

For each wdl:

- Extract the tool name from the file name (e.g. `bowtie` from `ww-bowtie.wdl`)
- Extract the tasks using regex to 'chunk' by task and then read the task name within that chunk

### 2. Query bio.tools using their [API to get metadata](https://biotools.readthedocs.io/en/latest/api_usage_guide.html#json) for each tool and store in a dataframe, with some exceptions:

- `ww-gatk.wdl`: Append `gatk_` to the task names and use those as tool names to query
- If we get `{'detail': 'Not found.'}` as a response then we must manually fill in metadata for those tools or repair the tool names as part of step 1. Some of our tools will not be found (e.g. `aws-sso`).

Information to get from bio.tools:

- Operations tool can perform (e.g. `'Sequence alignment'`)
- input data format(s)
- Output data format(s)
- Documentation URL


Then clean up the dataframe:

- FASTQ types specific to sequence platform (e.g. `'FASTQ-illumina'`, `''FASTQ-solexa'` should perhaps be collapsed to `'FASTQ'`)
- Remove duplicate rows

### 3. Add tasks and their metadata to a ChromaDB collection

- Do this using ChromaDB methods

### 4. Use bio.tools dataframe and ChromaDB collection downstream

- Use the dataframe to filter down CLI options based on what the user has input so far (e.g. only show tools that can perform an operation they selected)
- Use ChromaDB to get WDL task code chunks to supply to the LLM


In [92]:
import chromadb
import pandas as pd
import requests
import os
import re

## 1. Loop over WDLs

In [105]:
def chunk_by_task(wdl_text):
    chunks = {}
    lines = wdl_text.splitlines(keepends=True)
    i = 0
    while i < len(lines):
        match = re.match(r'^task\s+(\w+)\s*\{', lines[i])
        if match:
            task_name = match.group(1)
            depth = 0
            start = i
            while i < len(lines):
                depth += lines[i].count('{') - lines[i].count('}')
                i += 1
                if depth == 0:
                    break
            chunks[task_name] =  ''.join(lines[start:i])
        else:
            i += 1
    return chunks

def extract_wdl_tasks(wdl_dir):
    tool_tasks_dict = {}
    for wdl in os.listdir(wdl_dir):
        toolname = wdl[3:-4]
        with open(os.path.join(wdl_dir, wdl), "r") as f:
            content = f.read()
        tool_tasks_dict[toolname] = chunk_by_task(content)
    return tool_tasks_dict

# Get a dictionary of tools and their tasks
wdl_dir = '../data/wdl/'
tool_tasks_dict = extract_wdl_tasks(wdl_dir)

In [106]:
# Example tasks for a given tool:
tool_tasks_dict['cnvkit']

{'create_reference': 'task create_reference {\n  meta {\n    author: "Taylor Firman"\n    email: "tfirman@fredhutch.org"\n    description: "Create CNVkit reference from normal samples or pooled reference"\n    url: "https://raw.githubusercontent.com/getwilds/wilds-wdl-library/refs/heads/main/modules/ww-cnvkit/ww-cnvkit.wdl"\n    outputs: {\n        reference_cnn: "CNVkit reference file (.cnn)"\n    }\n  }\n\n  parameter_meta {\n    bam_files: "Array of BAM files for reference creation"\n    bam_indices: "Array of BAM index files corresponding to BAM files"\n    target_bed: "Target regions BED file"\n    antitarget_bed: "Antitarget regions BED file"\n    reference_fasta: "Reference genome FASTA file"\n    reference_fasta_index: "Reference genome FASTA index file"\n    cpu_cores: "Number of CPU cores to use"\n    memory_gb: "Memory allocation in GB"\n  }\n\n  input {\n    Array[File] bam_files\n    Array[File] bam_indices\n    File reference_fasta\n    File reference_fasta_index\n    Fil

## 2. Query bio.tools

In [149]:
orig_toolnames = list(tool_tasks_dict.keys())
orig_toolnames[:5]

['cnvkit', 'annotsv', 'cellranger', 'deseq2', 'bedparse']

In [150]:
# Repaired based on biotools naming conventions

toolnames = ['delly2' if x == 'delly' else x for x in orig_toolnames]

In [156]:
not_in_biotools = set()
need_manual_metadata = set()

tool_df_list = []

for tool in toolnames:
    if tool in {'bedparse', 'cellranger', 'ichorcna', 'testdata'}:
        not_in_biotools.add(tool)

    # Get tool information from bio.tools
    url = f"https://bio.tools/api/tool/{tool}?format=json" 
    biotools_data = requests.get(url).json()

    if biotools_data == {'detail': 'Not found.'}:
        not_in_biotools.add(tool)
        need_manual_metadata.add(tool)

    else:
        try:
            tool_df = pd.DataFrame()
            tool_df['operations'] = pd.json_normalize(biotools_data, 
                    record_path=['function', 'operation'])['term']
            tool_df['tool'] = tool
            tool_df['in_formats'] = pd.json_normalize(biotools_data, 
                    record_path=['function', 'input', 'format'])['term']
            tool_df['out_formats'] = pd.json_normalize(biotools_data, 
                    record_path=['function', 'output', 'format'])['term']
            tool_df['documentation'] = pd.json_normalize(biotools_data, 
                    record_path=['documentation'])['url']
            tool_df_list.append(tool_df)
        except KeyError:
            need_manual_metadata.add(tool)
            tool_df_list.append(tool_df)
            

In [157]:
not_in_biotools

{'aws-sso',
 'bedparse',
 'cellranger',
 'ichorcna',
 'jcast',
 'rmats-turbo',
 'shapemapper',
 'sjl',
 'smoove',
 'starling',
 'testdata',
 'tritonnp'}

In [158]:
df = pd.concat(tool_df_list)
df

,operations,tool,in_formats,out_formats,documentation
0,Variant calling,cnvkit,NaN,NaN,NaN
0,Annotation,annotsv,NaN,NaN,NaN
1,Genetic variation analysis,annotsv,NaN,NaN,NaN
0,Differential gene expression analysis,deseq2,DSV,DSV,http://www.bioconductor.org/packages/release/b...
1,RNA-Seq analysis,deseq2,NaN,NaN,NaN
...,...,...,...,...,...
6,Data sorting,samtools,NaN,NaN,NaN
0,Formatting,gdc,NaN,NaN,NaN
0,Genome assembly,megahit,NaN,NaN,NaN
0,Sequencing quality control,fastp,NaN,NaN,NaN


For reference, what the data coming from biotools looks like:

In [ ]:
biotools_data

{'name': 'Bowtie',
 'description': 'Bowtie is an ultrafast, memory-efficient short read aligner.',
 'homepage': 'http://bowtie-bio.sourceforge.net/index.shtml',
 'biotoolsID': 'bowtie',
 'biotoolsCURIE': 'biotools:bowtie',
 'version': ['1.2.2'],
 'otherID': [{'value': 'RRID:SCR_005476', 'type': 'rrid', 'version': None}],
 'relation': [],
 'function': [{'operation': [{'uri': 'http://edamontology.org/operation_0292',
     'term': 'Sequence alignment'}],
   'input': [{'data': {'uri': 'http://edamontology.org/data_0006',
      'term': 'Data'},
     'format': [{'uri': 'http://edamontology.org/format_2331', 'term': 'HTML'},
      {'uri': 'http://edamontology.org/format_1929', 'term': 'FASTA'}]},
    {'data': {'uri': 'http://edamontology.org/data_0006', 'term': 'Data'},
     'format': [{'uri': 'http://edamontology.org/format_1933',
       'term': 'FASTQ-solexa'},
      {'uri': 'http://edamontology.org/format_1932', 'term': 'FASTQ-sanger'},
      {'uri': 'http://edamontology.org/format_1931',
